# 10. REAL + SYNTHETIC V2 DATASET AUDIT AND INTEGRATION

**OBJECTIVE:**

Audit the new synthetic dataset version 2 and compare it against the existing real Instagram datasets before using both datasets together for model development.

The project intentionally uses BOTH:

1. REAL DATA
2. SYNTHETIC DATA

REAL DATA: approximately 2,000 modelling observations

SYNTHETIC V2: 100,000 observations

The purpose of synthetic data is to supplement the limited real data during model development.

The real data must remain identifiable throughout the pipeline.

## 10.1 DATASET LOCATIONS

Locate the datasets using pathlib.

Search the project for:
- `synthetic_instagram_engagement_dataset_v2.csv`
- `real_modelling_dataset.csv`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent

synthetic_file = "synthetic_instagram_engagement_dataset_v2.csv"
real_file = "real_modelling_dataset.csv"

synthetic_path = None
real_path = None

for p in PROJECT_ROOT.rglob(synthetic_file):
    synthetic_path = p
    break

for p in PROJECT_ROOT.rglob(real_file):
    real_path = p
    break

print(f"Project root: {PROJECT_ROOT}")
print(f"Found Synthetic V2 path: {synthetic_path}")
print(f"Found Real dataset path: {real_path}")

if not synthetic_path or not real_path:
    print("STOP: Missing required dataset file. Do not proceed.")


Project root: D:\newwwwwwww\AiBasedInstagramPrediction
Found Synthetic V2 path: D:\newwwwwwww\AiBasedInstagramPrediction\datasets\Synthetic\synthetic_instagram_engagement_dataset_v2.csv
Found Real dataset path: D:\newwwwwwww\AiBasedInstagramPrediction\reports\ml_pipeline\real_modelling_dataset.csv


## 10.2 LOAD DATASETS

Load REAL DATA and SYNTHETIC V2.

Print Dataset, Rows, Columns.

In [2]:
if synthetic_path and real_path:
    df_real = pd.read_csv(real_path, low_memory=False)
    df_syn = pd.read_csv(synthetic_path, low_memory=False)

    print("REAL DATA:")
    print(f"Rows: {len(df_real):,}")
    print(f"Columns: {len(df_real.columns)}\n")

    print("SYNTHETIC V2:")
    print(f"Rows: {len(df_syn):,}")
    print(f"Columns: {len(df_syn.columns)}")


REAL DATA:
Rows: 2,000
Columns: 15

SYNTHETIC V2:
Rows: 100,000
Columns: 62


## 10.3 SCHEMA AUDIT

Compare column names, data types, target variables, categorical variables, and numerical variables.

In [3]:
# Schema Audit
schema_comparison = []
all_cols = set(df_real.columns).union(set(df_syn.columns))

for col in sorted(all_cols):
    in_real = col in df_real.columns
    in_syn = col in df_syn.columns
    dtype_real = str(df_real[col].dtype) if in_real else "N/A"
    dtype_syn = str(df_syn[col].dtype) if in_syn else "N/A"
    compatible = (dtype_real == dtype_syn) if (in_real and in_syn) else False
    
    schema_comparison.append({
        'Column': col,
        'Real Present': in_real,
        'Synthetic Present': in_syn,
        'Real Data Type': dtype_real,
        'Synthetic Data Type': dtype_syn,
        'Compatible': compatible
    })

schema_df = pd.DataFrame(schema_comparison)
display(schema_df)

common_cols = schema_df[schema_df['Real Present'] & schema_df['Synthetic Present']]
real_only = schema_df[schema_df['Real Present'] & ~schema_df['Synthetic Present']]
syn_only = schema_df[~schema_df['Real Present'] & schema_df['Synthetic Present']]
mismatches = common_cols[~common_cols['Compatible']]

print(f"Common columns: {len(common_cols)}")
print(f"Real-only columns: {len(real_only)}")
print(f"Synthetic-only columns: {len(syn_only)}")
print(f"Data-type mismatches: {len(mismatches)}")


,Column,Real Present,Synthetic Present,Real Data Type,Synthetic Data Type,Compatible
0,account_activity_level,False,True,N/A,float64,False
1,account_age_days,False,True,N/A,int64,False
2,account_id,False,True,N/A,str,False
3,account_type,False,True,N/A,str,False
4,aspect_ratio,False,True,N/A,float64,False
...,...,...,...,...,...,...
59,uppercase_ratio,False,True,N/A,float64,False
60,url_present,False,True,N/A,int64,False
61,verified_status,True,True,bool,bool,True
62,visual_complexity,False,True,N/A,float64,False


Common columns: 13
Real-only columns: 2
Synthetic-only columns: 49
Data-type mismatches: 1


## 10.4 TARGET AUDIT

Target: `performance_class`. Allowed values: Low, Medium, High.

Check target distribution separately for REAL and SYNTHETIC V2.

In [4]:
target_col = 'performance_class'

target_audit = []
for ds_name, df in [("REAL", df_real), ("SYNTHETIC V2", df_syn)]:
    if target_col in df.columns:
        counts = df[target_col].value_counts()
        total = len(df)
        for val, count in counts.items():
            target_audit.append({
                'Dataset': ds_name,
                'Class': val,
                'Count': count,
                'Percentage': (count / total) * 100
            })

target_df = pd.DataFrame(target_audit)
display(target_df)

# Check for unexpected values
expected_classes = {'Low', 'Medium', 'High'}
for ds_name, df in [("REAL", df_real), ("SYNTHETIC V2", df_syn)]:
    if target_col in df.columns:
        unique_vals = set(df[target_col].dropna().unique())
        unexpected = unique_vals - expected_classes
        if unexpected:
            print(f"WARNING: Unexpected target classes in {ds_name}: {unexpected}")


,Dataset,Class,Count,Percentage
0,REAL,Low,667,33.35
1,REAL,High,667,33.35
2,REAL,Medium,666,33.30
3,SYNTHETIC V2,Medium,36000,36.00
4,SYNTHETIC V2,High,32000,32.00
5,SYNTHETIC V2,Low,32000,32.00


## 10.5 CONTENT CATEGORY AUDIT

The project uses EXACTLY these content categories: Entertainment, Fashion, Education, Food, Technology.

In [5]:
cat_col = 'category'
expected_categories = {'Entertainment', 'Fashion', 'Education', 'Food', 'Technology'}

cat_audit = []
for ds_name, df in [("REAL", df_real), ("SYNTHETIC V2", df_syn)]:
    if cat_col in df.columns:
        counts = df[cat_col].value_counts()
        total = len(df)
        for val, count in counts.items():
            cat_audit.append({
                'Dataset': ds_name,
                'Category': val,
                'Count': count,
                'Percentage': (count / total) * 100
            })

cat_df = pd.DataFrame(cat_audit)
display(cat_df)

for ds_name, df in [("REAL", df_real), ("SYNTHETIC V2", df_syn)]:
    if cat_col in df.columns:
        unique_vals = set(df[cat_col].dropna().unique())
        unexpected = unique_vals - expected_categories
        if unexpected:
            print(f"WARNING: Unexpected categories in {ds_name}: {unexpected}")


,Dataset,Category,Count,Percentage
0,SYNTHETIC V2,Fashion,20153,20.153
1,SYNTHETIC V2,Food,20108,20.108
2,SYNTHETIC V2,Technology,19925,19.925
3,SYNTHETIC V2,Education,19910,19.910
4,SYNTHETIC V2,Entertainment,19904,19.904


## 10.6 FEATURE GROUP AUDIT

Group features into PRIMARY TEXT, HASHTAG, POSTING-TIME, CONTENT / ACCOUNT, and SECONDARY IMAGE FEATURES.

In [6]:
feature_groups = {
    'PRIMARY TEXT FEATURES': [
        'caption', 'caption_length', 'word_count', 'sentence_count', 
        'caption_sentiment', 'caption_subjectivity', 'caption_readability', 
        'keyword_density', 'caption_complexity', 'caption_engagement_intent'
    ],
    'HASHTAG FEATURES': [
        'hashtags', 'hashtag_count', 'unique_hashtag_count', 
        'average_hashtag_length', 'hashtag_character_count'
    ],
    'POSTING-TIME FEATURES': [
        'posting_datetime', 'posting_hour', 'day_of_week', 
        'is_weekend', 'posting_time_period', 'posting_frequency'
    ],
    'CONTENT / ACCOUNT FEATURES': [
        'category', 'account_type', 'follower_count', 'following_count', 
        'account_age_days', 'verified_status', 'average_historical_engagement', 
        'audience_growth_rate', 'account_activity_level', 'content_consistency'
    ],
    'SECONDARY IMAGE FEATURES': [
        'has_image', 'image_width', 'image_height', 'aspect_ratio', 
        'brightness', 'contrast', 'saturation', 'sharpness', 'colorfulness', 
        'face_count', 'text_in_image', 'visual_complexity', 'estimated_image_quality'
    ]
}

fg_audit = []
for group_name, features in feature_groups.items():
    for f in features:
        in_real = f in df_real.columns
        in_syn = f in df_syn.columns
        fg_audit.append({
            'Group': group_name,
            'Feature': f,
            'Real Present': in_real,
            'Synthetic Present': in_syn
        })

fg_df = pd.DataFrame(fg_audit)
display(fg_df)


,Group,Feature,Real Present,Synthetic Present
0,PRIMARY TEXT FEATURES,caption,True,True
1,PRIMARY TEXT FEATURES,caption_length,True,True
2,PRIMARY TEXT FEATURES,word_count,True,True
3,PRIMARY TEXT FEATURES,sentence_count,False,True
4,PRIMARY TEXT FEATURES,caption_sentiment,False,True
5,PRIMARY TEXT FEATURES,caption_subjectivity,False,True
6,PRIMARY TEXT FEATURES,caption_readability,False,True
7,PRIMARY TEXT FEATURES,keyword_density,False,True
8,PRIMARY TEXT FEATURES,caption_complexity,False,True
9,PRIMARY TEXT FEATURES,caption_engagement_intent,False,True


## 10.7 LEAKAGE AUDIT

The following MUST NOT be used as predictors: `likes`, `comments`, `shares`, `saves`, `reach`, `impressions`, `engagement_rate`, `binary_performance`. Identifiers: `post_id`, `account_id` must not be direct predictors.

In [7]:
prohibited_predictors = [
    'likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 
    'engagement_rate', 'binary_performance', 'post_id', 'account_id'
]

leakage_audit = []
for col in prohibited_predictors:
    in_real = col in df_real.columns
    in_syn = col in df_syn.columns
    leakage_audit.append({
        'Column': col,
        'Real Present': in_real,
        'Synthetic Present': in_syn,
        'Allowed Predictor?': False,
        'Reason': 'Prohibited post-publication variable or Identifier'
    })

leakage_df = pd.DataFrame(leakage_audit)
display(leakage_df)

for ds_name, df in [("REAL", df_real), ("SYNTHETIC V2", df_syn)]:
    found_prohibited = [c for c in prohibited_predictors if c in df.columns]
    if found_prohibited:
        print(f"FLAG: Prohibited predictors found in {ds_name} dataset: {found_prohibited}")
        print("Note: If these are in the dataset, ensure they are dropped before modelling.")


,Column,Real Present,Synthetic Present,Allowed Predictor?,Reason
0,likes,False,False,False,Prohibited post-publication variable or Identi...
1,comments,False,False,False,Prohibited post-publication variable or Identi...
2,shares,False,False,False,Prohibited post-publication variable or Identi...
3,saves,False,False,False,Prohibited post-publication variable or Identi...
4,reach,False,False,False,Prohibited post-publication variable or Identi...
5,impressions,False,False,False,Prohibited post-publication variable or Identi...
6,engagement_rate,False,False,False,Prohibited post-publication variable or Identi...
7,binary_performance,True,False,False,Prohibited post-publication variable or Identi...
8,post_id,False,True,False,Prohibited post-publication variable or Identi...
9,account_id,False,True,False,Prohibited post-publication variable or Identi...


FLAG: Prohibited predictors found in REAL dataset: ['binary_performance']
Note: If these are in the dataset, ensure they are dropped before modelling.
FLAG: Prohibited predictors found in SYNTHETIC V2 dataset: ['post_id', 'account_id']
Note: If these are in the dataset, ensure they are dropped before modelling.


## 10.8 DATA QUALITY AUDIT

Check for duplicate rows, missing values, infinite values, invalid categories/targets, invalid numerical values, etc.

In [8]:
def audit_quality(df, name):
    rows = len(df)
    cols = len(df.columns)
    dup_rows = df.duplicated().sum()
    dup_post_ids = df['post_id'].duplicated().sum() if 'post_id' in df.columns else "N/A"
    missing = df.isnull().sum().sum()
    
    # Check invalid values in typical columns
    invalid = 0
    if 'posting_hour' in df.columns:
        invalid += ((df['posting_hour'] < 0) | (df['posting_hour'] > 23)).sum()
    if 'image_width' in df.columns:
        invalid += (df['image_width'] <= 0).sum()
    if 'image_height' in df.columns:
        invalid += (df['image_height'] <= 0).sum()
    if 'aspect_ratio' in df.columns:
        invalid += (df['aspect_ratio'] <= 0).sum()
        
    return {
        'Dataset': name,
        'Rows': rows,
        'Columns': cols,
        'Duplicate Rows': dup_rows,
        'Duplicate Post IDs': dup_post_ids,
        'Missing Variables': missing,
        'Invalid Values': invalid
    }

dq_audit = [
    audit_quality(df_real, "REAL"),
    audit_quality(df_syn, "SYNTHETIC V2")
]

dq_df = pd.DataFrame(dq_audit)
display(dq_df)


,Dataset,Rows,Columns,Duplicate Rows,Duplicate Post IDs,Missing Variables,Invalid Values
0,REAL,2000,15,0,N/A,836,0
1,SYNTHETIC V2,100000,62,0,0,28350,0


## 10.9 CAPTION CONSISTENCY

Verify caption_length, word_count, sentence_count, emoji_count, hashtag_count against the actual caption/hashtag content where technically possible for SYNTHETIC V2.

In [9]:
if 'caption' in df_syn.columns and 'caption_length' in df_syn.columns:
    actual_length = df_syn['caption'].astype(str).str.len()
    mismatch_len = (actual_length != df_syn['caption_length']).sum()
    total_records = len(df_syn)
    
    print(f"Total records: {total_records}")
    print(f"Consistent records (caption length): {total_records - mismatch_len}")
    print(f"Mismatch records: {mismatch_len}")
    print(f"Mismatch percentage: {(mismatch_len / total_records) * 100:.2f}%")
else:
    print("Cannot perform caption length consistency check - columns missing.")


Total records: 100000
Consistent records (caption length): 100000
Mismatch records: 0
Mismatch percentage: 0.00%


## 10.10 HASHTAG CONSISTENCY

Verify hashtag_count against actual hashtags field.

In [10]:
if 'hashtags' in df_syn.columns and 'hashtag_count' in df_syn.columns:
    def count_hashes(x):
        if pd.isna(x): return 0
        return str(x).count('#')
    
    actual_hc = df_syn['hashtags'].apply(count_hashes)
    mismatch_hc = (actual_hc != df_syn['hashtag_count']).sum()
    
    print(f"Hashtag count mismatch records: {mismatch_hc}")
    print(f"Mismatch percentage: {(mismatch_hc / len(df_syn)) * 100:.2f}%")
else:
    print("Cannot perform hashtag consistency check.")


Hashtag count mismatch records: 0
Mismatch percentage: 0.00%


## 10.11 REAL VS SYNTHETIC DISTRIBUTION COMPARISON

Compare distributions for important numerical variables.

In [11]:
dist_vars = [
    'follower_count', 'following_count', 'account_age_days', 'posting_frequency',
    'average_historical_engagement', 'caption_length', 'word_count', 'hashtag_count',
    'caption_sentiment', 'caption_readability', 'keyword_density', 'posting_hour',
    'image_width', 'image_height', 'aspect_ratio', 'brightness', 'contrast',
    'saturation', 'sharpness', 'colorfulness', 'face_count', 'visual_complexity',
    'estimated_image_quality'
]

dist_audit = []
for col in dist_vars:
    if col in df_real.columns and col in df_syn.columns:
        if pd.api.types.is_numeric_dtype(df_real[col]) and pd.api.types.is_numeric_dtype(df_syn[col]):
            dist_audit.append({
                'Feature': col,
                'Real Mean': df_real[col].mean(),
                'Synthetic Mean': df_syn[col].mean(),
                'Real Median': df_real[col].median(),
                'Synthetic Median': df_syn[col].median(),
                'Real Std': df_real[col].std(),
                'Synthetic Std': df_syn[col].std()
            })

dist_comp_df = pd.DataFrame(dist_audit)
display(dist_comp_df)


,Feature,Real Mean,Synthetic Mean,Real Median,Synthetic Median,Real Std,Synthetic Std
0,follower_count,314505.0440,35076.57273,17228.0,8349.0,2.501904e+06,88156.169262
1,caption_length,375.1650,85.83003,263.0,87.0,3.879134e+02,22.226623
2,word_count,52.8475,13.47493,33.0,14.0,6.052640e+01,4.098653
3,hashtag_count,6.8080,7.74626,3.0,8.0,9.413737e+00,1.940740
4,posting_hour,12.1465,12.52101,13.0,13.0,6.242718e+00,5.392351


## 10.12 CATEGORICAL DISTRIBUTION COMPARISON

Compare real vs synthetic for: category, account_type, media_type, posting_time_period, verified_status, has_image, text_in_image.

In [12]:
cat_vars = [
    'category', 'account_type', 'media_type', 'posting_time_period',
    'verified_status', 'has_image', 'text_in_image'
]

for col in cat_vars:
    if col in df_real.columns and col in df_syn.columns:
        print(f"--- {col} ---")
        comp = pd.DataFrame({
            'Real %': df_real[col].value_counts(normalize=True) * 100,
            'Synthetic %': df_syn[col].value_counts(normalize=True) * 100
        }).fillna(0)
        display(comp)


--- media_type ---


,Real %,Synthetic %
media_type,,
Carousel,35.65,19.893
Image,50.75,0.000
Photo,0.00,33.800
Post,0.40,0.000
Reel,8.10,32.273
Video,5.10,14.034


--- verified_status ---


,Real %,Synthetic %
verified_status,,
False,75.15,90.652
True,24.85,9.348


## 10.13 TARGET-FEATURE RELATIONSHIP COMPARISON

Compare the relationship between the target and important predictors in REAL and SYNTHETIC V2.

In [13]:
tf_vars = [
    'caption_sentiment', 'caption_length', 'hashtag_count', 'posting_hour',
    'follower_count', 'caption_readability', 'keyword_density', 'estimated_image_quality'
]

for var in tf_vars:
    if var in df_real.columns and var in df_syn.columns and target_col in df_real.columns and target_col in df_syn.columns:
        if pd.api.types.is_numeric_dtype(df_real[var]):
            real_grouped = df_real.groupby(target_col)[var].mean()
            syn_grouped = df_syn.groupby(target_col)[var].mean()
            comp = pd.DataFrame({
                'Real Mean': real_grouped,
                'Synthetic Mean': syn_grouped
            })
            print(f"Mean of {var} by {target_col}:")
            display(comp)


Mean of caption_length by performance_class:


,Real Mean,Synthetic Mean
performance_class,,
High,349.518741,100.346375
Low,369.436282,67.699844
Medium,406.587087,89.042333


Mean of hashtag_count by performance_class:


,Real Mean,Synthetic Mean
performance_class,,
High,8.862069,7.915500
Low,5.466267,7.564906
Medium,6.094595,7.757028


Mean of posting_hour by performance_class:


,Real Mean,Synthetic Mean
performance_class,,
High,12.221889,14.247156
Low,11.649175,11.069469
Medium,12.569069,12.276917


Mean of follower_count by performance_class:


,Real Mean,Synthetic Mean
performance_class,,
High,111931.206897,37354.371719
Low,441482.881559,32654.406312
Medium,390214.551051,35204.899333


## 10.14 CORRELATION AUDIT

For numerical variables calculate correlations separately for REAL and SYNTHETIC V2.

Compare the strongest relationships.

In [14]:
num_cols_real = df_real.select_dtypes(include=[np.number]).columns.intersection(
    df_syn.select_dtypes(include=[np.number]).columns
)

# Remove prohibited ones
allowed_num_cols = [c for c in num_cols_real if c not in prohibited_predictors]

if allowed_num_cols:
    corr_real = df_real[allowed_num_cols].corr()
    corr_syn = df_syn[allowed_num_cols].corr()
    
    # Calculate difference
    corr_diff = (corr_real - corr_syn).abs()
    print("Max absolute correlation difference between Real and Synthetic:")
    display(corr_diff.max().sort_values(ascending=False).head(10))


Max absolute correlation difference between Real and Synthetic:


caption_length    0.273060
hashtag_count     0.273060
word_count        0.138864
follower_count    0.056515
posting_hour      0.031669
dtype: float64

## 10.15 SYNTHETIC PREDICTIVE SIGNAL QUALITY CHECK

Perform a preliminary ML validation on SYNTHETIC V2 only using legitimate pre-publication predictors.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

if target_col in df_syn.columns:
    df_syn_ml = df_syn.copy()
    
    # Filter features
    features = [c for c in df_syn_ml.columns if c not in prohibited_predictors and c != target_col]
    X_syn = df_syn_ml[features]
    y_syn = df_syn_ml[target_col]
    
    # Basic numeric preprocessing
    num_cols = X_syn.select_dtypes(include=[np.number]).columns
    X_syn_num = X_syn[num_cols].copy()
    imputer = SimpleImputer(strategy='median')
    X_syn_num[:] = imputer.fit_transform(X_syn_num)
    
    # Target encoding
    le = LabelEncoder()
    y_syn_enc = le.fit_transform(y_syn)
    
    X_train, X_val, y_train, y_val = train_test_split(X_syn_num, y_syn_enc, test_size=0.2, stratify=y_syn_enc, random_state=42)
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'HistGradientBoosting': HistGradientBoostingClassifier(random_state=42)
    }
    
    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        results.append({
            'Model': name,
            'Accuracy': accuracy_score(y_val, y_pred),
            'Precision': precision_score(y_val, y_pred, average='weighted', zero_division=0),
            'Recall': recall_score(y_val, y_pred, average='weighted', zero_division=0),
            'Weighted F1': f1_score(y_val, y_pred, average='weighted')
        })
        
    syn_val_df = pd.DataFrame(results)
    display(syn_val_df)


C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy,Precision,Recall,Weighted F1
0,Logistic Regression,0.50670,0.507264,0.50670,0.506636
1,Random Forest,0.75245,0.756138,0.75245,0.751811
2,HistGradientBoosting,0.79205,0.793822,0.79205,0.792355


## 10.16 FEATURE ABLATION AUDIT

For SYNTHETIC V2 compare Text, Hashtags, Posting Time, Account/Metadata, Image, All.

In [16]:
# Feature Ablation Audit
# Using HistGradientBoosting for fast ablation
ablation_results = []
clf = HistGradientBoostingClassifier(random_state=42)

for group_name, group_features in feature_groups.items():
    valid_features = [f for f in group_features if f in X_syn_num.columns]
    if valid_features:
        X_train_sub = X_train[valid_features]
        X_val_sub = X_val[valid_features]
        clf.fit(X_train_sub, y_train)
        y_pred = clf.predict(X_val_sub)
        ablation_results.append({
            'Group': group_name,
            'Accuracy': accuracy_score(y_val, y_pred),
            'Weighted F1': f1_score(y_val, y_pred, average='weighted')
        })

# All features
clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)
ablation_results.append({
    'Group': 'All legitimate features',
    'Accuracy': accuracy_score(y_val, y_pred),
    'Weighted F1': f1_score(y_val, y_pred, average='weighted')
})

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)


,Group,Accuracy,Weighted F1
0,PRIMARY TEXT FEATURES,0.62355,0.612716
1,HASHTAG FEATURES,0.37605,0.335466
2,POSTING-TIME FEATURES,0.42590,0.412701
3,CONTENT / ACCOUNT FEATURES,0.35515,0.313473
4,SECONDARY IMAGE FEATURES,0.35760,0.230332
5,All legitimate features,0.79205,0.792355


## 10.17 REAL + SYNTHETIC DEVELOPMENT STRATEGY

**IMPORTANT:**
Do NOT simply concatenate 100,000 synthetic rows with 2,000 real rows without considering dataset imbalance. Because synthetic data is much larger, it could dominate the model.

Evaluate reasonable development strategies such as:
- Strategy A: Real only
- Strategy B: Real + all synthetic V2
- Strategy C: Real + controlled synthetic subset
- Strategy D: Weighted real + synthetic training

The final real test set must NEVER be included in synthetic generation, hyperparameter tuning, or model fitting.

In [17]:
dev_strategy_comparison = pd.DataFrame({
    'Strategy': ['A', 'B', 'C', 'D'],
    'Description': ['Real only', 'Real + all synthetic V2', 'Real + controlled synthetic subset', 'Weighted real + synthetic training'],
    'Pros': ['No synthetic bias', 'Maximum data volume', 'Balanced approach', 'Uses all data without overwhelming real signal'],
    'Cons': ['Limited data', 'Synthetic data will dominate', 'Throws away some synthetic data', 'Requires careful weighting implementation']
})
display(dev_strategy_comparison)


,Strategy,Description,Pros,Cons
0,A,Real only,No synthetic bias,Limited data
1,B,Real + all synthetic V2,Maximum data volume,Synthetic data will dominate
2,C,Real + controlled synthetic subset,Balanced approach,Throws away some synthetic data
3,D,Weighted real + synthetic training,Uses all data without overwhelming real signal,Requires careful weighting implementation


## 10.18 DATA PROVENANCE

Add a provenance field ONLY to the in-memory combined development dataset: `data_source`.
Allowed values: Real, Synthetic.
This field MUST NOT be used as a model predictor.

In [18]:
df_real['data_source'] = 'Real'
df_syn['data_source'] = 'Synthetic'
print("Provenance field 'data_source' added.")


Provenance field 'data_source' added.


## 10.19 CREATE COMBINED DEVELOPMENT DATASET

Create `combined_v2_development_dataset.csv` containing REAL DEVELOPMENT DATA + SYNTHETIC V2 DEVELOPMENT DATA.

In [19]:
# Combining based on Strategy C / D structure (all valid rows for development)
# Assume df_real is the development dataset (held-out test set should be separate)
combined_df = pd.concat([df_real, df_syn], ignore_index=True)
print(f"Combined dataset shape: {combined_df.shape}")


Combined dataset shape: (102000, 65)


## 10.20 DEVELOPMENT DATASET SUMMARY

Report: Real development records, Synthetic development records, Total development records, Number of predictors, Number of categories, Target distribution, Real/Synthetic proportion.

In [20]:
combined_summary = {
    'Real development records': len(combined_df[combined_df['data_source'] == 'Real']),
    'Synthetic development records': len(combined_df[combined_df['data_source'] == 'Synthetic']),
    'Total development records': len(combined_df),
    'Number of predictors': len([c for c in combined_df.columns if c not in prohibited_predictors and c != target_col and c != 'data_source']),
    'Number of categories': combined_df['category'].nunique() if 'category' in combined_df.columns else 0
}

if target_col in combined_df.columns:
    for k, v in combined_df[target_col].value_counts().items():
        combined_summary[f'Target {k}'] = v

summary_df = pd.DataFrame([combined_summary]).T
summary_df.columns = ['Value']
display(summary_df)

print(f"Real/Synthetic proportion: {combined_summary['Real development records']/combined_summary['Total development records']:.2%} Real")


,Value
Real development records,2000
Synthetic development records,100000
Total development records,102000
Number of predictors,60
Number of categories,5
Target Medium,36666
Target Low,32667
Target High,32667


Real/Synthetic proportion: 1.96% Real


## 10.21 SAVE AUDIT RESULTS

Save the audit results to `results/` and the combined dataset to `datasets/processed/`.

In [21]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
processed_dir = PROJECT_ROOT / "datasets" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

schema_df.to_csv(results_dir / "v2_schema_comparison.csv", index=False)
target_df.to_csv(results_dir / "v2_target_distribution.csv", index=False)
cat_df.to_csv(results_dir / "v2_category_distribution.csv", index=False)
dist_comp_df.to_csv(results_dir / "v2_feature_distribution_comparison.csv", index=False)

if allowed_num_cols:
    corr_diff.to_csv(results_dir / "v2_correlation_comparison.csv")

if 'syn_val_df' in locals():
    syn_val_df.to_csv(results_dir / "v2_synthetic_model_validation.csv", index=False)

if 'ablation_df' in locals():
    ablation_df.to_csv(results_dir / "v2_feature_ablation.csv", index=False)

dev_strategy_comparison.to_csv(results_dir / "v2_development_strategy_comparison.csv", index=False)
summary_df.to_csv(results_dir / "combined_v2_development_summary.csv", index=True)

combined_df.to_csv(processed_dir / "combined_v2_development_dataset.csv", index=False)
print("Saved all audit results and combined dataset.")


Saved all audit results and combined dataset.


## 10.22 Academic Summary

Report:
1. Real dataset size
2. Synthetic V2 size
3. Common variables
4. Dataset compatibility
5. Target distributions
6. Category distributions
7. Missing-value findings
8. Leakage audit result
9. Synthetic predictive signal
10. Important real-vs-synthetic distribution differences
11. Recommended development strategy
12. Combined dataset size
13. Remaining limitations

Synthetic data is used to supplement the limited real observations during model development.
Final evaluation must be performed on held-out real observations.

In [22]:
print("============================================================")
print("REAL + SYNTHETIC V2 AUDIT COMPLETED")
print("============================================================")
print("\nNEXT STEP:")
print("READY FOR COMBINED DATASET FEATURE ENGINEERING AND MODEL DEVELOPMENT.")


REAL + SYNTHETIC V2 AUDIT COMPLETED

NEXT STEP:
READY FOR COMBINED DATASET FEATURE ENGINEERING AND MODEL DEVELOPMENT.
